In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_ENDPOINT = os.environ.get("DATABRICKS_ENDPOINT")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

## Function Calling Example

1. define one local function and tool schema
2. ask a question that should trigger the tool
3. execute the tool call
4. send tool result back for the final answer

In [ ]:
import json


def add_numbers(a: int, b: int) -> int:
    return a + b


tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    }
]

messages = [
    {"role": "user", "content": "What is 17 + 25? Use the tool."}
]

first = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)
first.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_a6546f09-dbcb-4bac-ad7c-05d43fe53d11', function=Function(arguments='{"a": 17, "b": 25}', name='add_numbers'), type='function')])

In [9]:

assistant_msg = first.choices[0].message

# Keep only fields needed by the API for the assistant tool-call message
messages.append(
    {
        "role": "assistant",
        "content": assistant_msg.content or "",
        "tool_calls": [tc.model_dump() for tc in (assistant_msg.tool_calls or [])],
    }
)

if assistant_msg.tool_calls:
    tc = assistant_msg.tool_calls[0]
    args = json.loads(tc.function.arguments)
    result = add_numbers(**args)

    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps({"result": result}),
        }
    )

    final = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        tools=tools,
        temperature=0,
    )

    print("Tool called:", tc.function.name)
    print("Arguments:", args)
    print("Function result:", result)
    print("Final answer:", final.choices[0].message.content)
else:
    print("Model did not call a tool. Re-run to try again.")

Tool called: add_numbers
Arguments: {'a': 17, 'b': 25}
Function result: -8
Final answer: None
